# Chapter 3: AutoGluon — Maximum Accuracy Through Ensembling

### Why This Matters

Where FLAML optimizes for speed-per-accuracy, **AutoGluon** (from AWS) optimizes primarily for raw predictive accuracy, largely through aggressive automatic **stacking** (training many diverse models, then training a further "meta-model" that learns how to best combine their predictions) — and it's known for delivering unusually strong results with minimal configuration, at the cost of longer training time and larger resulting model size.

In [ ]:
from autogluon.tabular import TabularPredictor

predictor = TabularPredictor(label="churn", eval_metric="roc_auc").fit(
    train_df,
    time_limit=120,                # seconds
    presets="medium_quality"       # or "best_quality" for a longer, more thorough (and much slower) run
)

leaderboard = predictor.leaderboard(test_df, silent=True)
print(leaderboard.head())

preds_proba = predictor.predict_proba(test_df)

Notice how little code this is relative to everything you manually assembled in Volumes 05–06: no manual train/test split management within the call, no manual encoding of `contract`/`internet`/`tech_support`, no manual model loop — `TabularPredictor` handles the entire pipeline internally, including trying multiple model families (neural nets, gradient boosting, random forests, and more) and stacking their predictions. `presets="medium_quality"` vs `"best_quality"` is AutoGluon's explicit speed/accuracy dial — start with `medium_quality` for initial exploration, and reserve `best_quality` (which can take substantially longer) for a final production candidate once you're confident the problem framing is right.

⚠️ **The tradeoff for this convenience and accuracy is deployment complexity.** A stacked ensemble of many models is inherently harder to explain to a non-technical stakeholder than the single decision tree from Volume 05, and it's a heavier, slower artifact to serve in production than a single XGBoost model. Reach for AutoGluon specifically when raw predictive accuracy is the dominant business priority and you have the infrastructure to serve a more complex model — not as an automatic default over the simpler tools in Volumes 05–06.

### Cheat Sheet — AutoGluon

| Task | Code |
|---|---|
| Train | `TabularPredictor(label="target", eval_metric="roc_auc").fit(train_df, time_limit=120)` |
| See what won | `predictor.leaderboard(test_df)` |
| Predict | `predictor.predict_proba(test_df)` |
| Speed vs. accuracy | `presets="medium_quality"` vs. `"best_quality"` |

---